# Arg KPI USD Government Product

**Autor:** Renzo Gutierrez  
**Data product:** evolución del dólar oficial explicada por presidencias y gestiones económicas

## Objetivo

Esta notebook integra dos productos construidos previamente:

1. `Arg_KPI_USD`: cotización histórica oficial del dólar publicada por el BCRA.
2. `Arg_KPI_Government`: dimensión diaria de presidentes y ministros del área económica.

El resultado es una **capa Gold** lista para consumo analítico. Además de conservar el detalle diario, genera dos tablas agregadas y dos waterfall charts:

- contribución a la variación del dólar por presidente;
- contribución a la variación del dólar por ministro de Economía o responsable principal de la cartera económica.

### Decisión analítica principal

La variación se mide en pesos argentinos por dólar. Se parte de la primera cotización disponible, se calcula la diferencia contra la cotización anterior y esa diferencia se asigna a la autoridad vigente en la fecha de la nueva cotización. Por construcción:

`cotización inicial + suma de contribuciones = cotización actual`

Esta es una **atribución temporal descriptiva**, útil para analizar períodos y construir visualizaciones. No demuestra que una autoridad haya causado por sí sola una variación, porque el tipo de cambio también responde a factores monetarios, fiscales, regulatorios e internacionales.


## 1. Arquitectura y decisiones técnicas

| Capa | Tabla | Grano | Uso |
|---|---|---|---|
| Bronze | `usd_quotes` | Una cotización por fecha informada por el BCRA | Fuente económica |
| Silver | `government_context_daily` | Una fila por día calendario | Contexto político |
| Gold | `usd_government_daily` | Una fila por fecha con cotización | Dataset integrado |
| Gold | `usd_variation_by_president` | Una fila por presidente | Waterfall presidencial |
| Gold | `usd_variation_by_economy_minister` | Una fila por ministro | Waterfall ministerial |

El join se realiza por `date`. La serie del BCRA no necesariamente posee observaciones los fines de semana y feriados; por eso la capa Gold conserva solamente las fechas con cotización. Cuando existe un salto entre dos observaciones, la diferencia se atribuye a la autoridad vigente en la fecha de la cotización nueva.

Las tablas Gold se reconstruyen con `overwrite` porque son datasets derivados y reproducibles. Luego se ejecuta `VACUUM` con siete días de retención para eliminar archivos obsoletos sin comprometer el historial activo de Delta Lake.


## 2. Dependencias

Se utiliza `deltalake` para leer y escribir las tablas en MinIO, `pandas` para la transformación y `plotly` para generar waterfall charts interactivos.


In [ ]:
%pip install -q pandas pyarrow deltalake plotly

In [ ]:
import os
from datetime import date
from getpass import getpass
from typing import Any

import pandas as pd
import plotly.graph_objects as go
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 3. Configuración y rutas

La notebook reutiliza las credenciales de MinIO empleadas en los pipelines anteriores. No se guardan secretos en el archivo: se leen desde variables de entorno o se solicitan de forma interactiva.

Las rutas de entrada conservan el nombre de su fuente y la salida utiliza `arg_usd_government_product`, que describe el data product creado.


In [ ]:
VACUUM_RETENTION_HOURS = 168  # 7 días


def get_setting(name: str, prompt: str, secret: bool = False) -> str:
    """Obtiene una configuración del entorno o la solicita al usuario."""
    value = os.getenv(name)
    if value:
        return value
    return getpass(prompt) if secret else input(prompt).strip()


MINIO_ENDPOINT = get_setting("MINIO_ENDPOINT", "Endpoint de MinIO: ")
MINIO_ACCESS_KEY = get_setting("MINIO_ACCESS_KEY", "Access key de MinIO: ", secret=True)
MINIO_SECRET_KEY = get_setting("MINIO_SECRET_KEY", "Secret key de MinIO: ", secret=True)
MINIO_BUCKET = get_setting("MINIO_BUCKET", "Bucket de MinIO: ")

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_ALLOW_HTTP": "true",
    "aws_conditional_put": "etag",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

# Entradas producidas por las notebooks anteriores.
USD_QUOTES_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/bronze/"
    "bcra_estadisticas_cambiarias_api/usd_quotes"
)
GOVERNMENT_DAILY_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/silver/"
    "arg_government_context/government_context_daily"
)

# Salidas analíticas de esta notebook.
GOLD_BASE_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/gold/arg_usd_government_product"
)
USD_GOVERNMENT_DAILY_PATH = f"{GOLD_BASE_PATH}/usd_government_daily"
USD_BY_PRESIDENT_PATH = f"{GOLD_BASE_PATH}/usd_variation_by_president"
USD_BY_MINISTER_PATH = f"{GOLD_BASE_PATH}/usd_variation_by_economy_minister"

print(f"Entrada USD: {USD_QUOTES_PATH}")
print(f"Entrada Gobierno: {GOVERNMENT_DAILY_PATH}")
print(f"Salida Gold: {GOLD_BASE_PATH}")

## 4. Lectura de las tablas Delta

Esta etapa es la **E** de un proceso ETL sobre datos internos: en lugar de volver a consultar las APIs, consume los productos Bronze y Silver ya validados. Esto evita duplicar lógica de extracción y mantiene responsabilidades claras entre notebooks.


In [ ]:
def read_delta_table(data_path: str) -> pd.DataFrame:
    """Lee la versión activa de una tabla Delta almacenada en MinIO."""
    try:
        return DeltaTable(
            data_path,
            storage_options=storage_options,
        ).to_pandas()
    except TableNotFoundError as error:
        raise FileNotFoundError(
            f"No se encontró la tabla Delta: {data_path}. "
            "Ejecutá primero las notebooks upstream."
        ) from error


def extract_source_products() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Obtiene los dos productos que alimentan la capa Gold."""
    usd_quotes = read_delta_table(USD_QUOTES_PATH)
    government_daily = read_delta_table(GOVERNMENT_DAILY_PATH)
    return usd_quotes, government_daily

## 5. Validación de contratos de entrada

Antes del join se comprueba que cada tabla mantenga su contrato mínimo: columnas obligatorias, clave única por fecha y valores válidos. Si un pipeline anterior cambia de esquema, esta notebook falla temprano con un mensaje claro.


In [ ]:
def validate_source_products(
    usd_quotes: pd.DataFrame,
    government_daily: pd.DataFrame,
) -> None:
    """Valida los contratos esperados de las capas Bronze y Silver."""
    usd_required = {"date", "exchange_rate_ars"}
    government_required = {
        "date",
        "president",
        "economy_minister",
        "political_space",
        "mandate_type",
    }

    missing_usd = usd_required.difference(usd_quotes.columns)
    missing_government = government_required.difference(government_daily.columns)

    if missing_usd:
        raise ValueError(f"Faltan columnas en USD: {sorted(missing_usd)}")
    if missing_government:
        raise ValueError(
            f"Faltan columnas en gobierno: {sorted(missing_government)}"
        )
    if usd_quotes.empty or government_daily.empty:
        raise ValueError("Una de las tablas de entrada está vacía.")
    if usd_quotes["date"].duplicated().any():
        raise ValueError("La tabla USD contiene fechas duplicadas.")
    if government_daily["date"].duplicated().any():
        raise ValueError("La dimensión de gobierno contiene fechas duplicadas.")
    if (pd.to_numeric(usd_quotes["exchange_rate_ars"], errors="coerce") <= 0).any():
        raise ValueError("La tabla USD contiene cotizaciones no positivas.")

## 6. Construcción del data product diario

La transformación normaliza las fechas, selecciona las columnas de negocio y ejecuta un join `one_to_one`. Luego calcula:

- cotización anterior;
- variación diaria absoluta en ARS;
- variación porcentual diaria;
- fecha y hora de actualización del producto.

La primera observación recibe una variación de cero porque su valor se representa como el punto de partida del waterfall.


In [ ]:
def build_usd_government_daily(
    usd_quotes: pd.DataFrame,
    government_daily: pd.DataFrame,
) -> pd.DataFrame:
    """Integra cotizaciones y autoridades, y calcula los cambios diarios."""
    usd = usd_quotes.copy()
    government = government_daily.copy()

    usd["date"] = pd.to_datetime(usd["date"], errors="coerce").dt.normalize()
    government["date"] = pd.to_datetime(
        government["date"], errors="coerce"
    ).dt.normalize()
    usd["exchange_rate_ars"] = pd.to_numeric(
        usd["exchange_rate_ars"], errors="coerce"
    )

    usd_columns = [
        column
        for column in [
            "date",
            "currency_code",
            "currency_name",
            "exchange_rate_ars",
            "pass_through_rate",
        ]
        if column in usd.columns
    ]
    government_columns = [
        column
        for column in [
            "date",
            "president",
            "political_space",
            "mandate_type",
            "presidential_term",
            "economy_minister",
            "ministry_title",
            "government_context",
            "president_change_flag",
            "minister_change_flag",
            "government_change_flag",
        ]
        if column in government.columns
    ]

    fact = (
        usd[usd_columns]
        .dropna(subset=["date", "exchange_rate_ars"])
        .sort_values("date")
        .merge(
            government[government_columns],
            on="date",
            how="left",
            validate="one_to_one",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    fact["previous_exchange_rate_ars"] = fact["exchange_rate_ars"].shift(1)
    fact["daily_change_ars"] = fact["exchange_rate_ars"].diff().fillna(0.0)
    fact["daily_change_pct"] = fact["exchange_rate_ars"].pct_change() * 100
    fact["is_first_observation"] = fact.index == 0
    fact["product_refreshed_at_utc"] = pd.Timestamp.now(tz="UTC")
    fact["attribution_method"] = (
        "Cambio vs. cotización anterior asignado a la autoridad vigente "
        "en la fecha de la nueva cotización"
    )

    return fact

## 7. Calidad del join y reconciliación diaria

El producto se detiene si alguna cotización queda sin presidente o ministro. También verifica que la suma de todos los cambios diarios reproduzca exactamente la diferencia entre la última y la primera cotización, tolerando únicamente errores mínimos de punto flotante.


In [ ]:
def validate_daily_product(fact: pd.DataFrame, tolerance: float = 1e-8) -> None:
    """Controla integridad del join y reconciliación de la serie."""
    if fact.empty:
        raise ValueError("El producto diario quedó vacío.")
    if fact["date"].duplicated().any():
        raise ValueError("El producto diario contiene fechas duplicadas.")

    for column in ["president", "economy_minister"]:
        if fact[column].isna().any():
            missing_dates = fact.loc[fact[column].isna(), "date"].head().tolist()
            raise ValueError(
                f"Existen cotizaciones sin {column}: {missing_dates}. "
                "Actualizá primero Arg_KPI_Government."
            )

    expected_change = (
        fact["exchange_rate_ars"].iloc[-1]
        - fact["exchange_rate_ars"].iloc[0]
    )
    calculated_change = fact["daily_change_ars"].sum()

    if abs(expected_change - calculated_change) > tolerance:
        raise ValueError(
            "La variación diaria no reconcilia con los niveles inicial y final."
        )

## 8. Agregación de contribuciones

Cada tabla agregada conserva el orden cronológico de las gestiones y muestra:

- primera y última fecha con cotización;
- cantidad de observaciones;
- nivel de apertura de referencia;
- nivel de cierre;
- contribución absoluta en ARS;
- variación porcentual dentro del período;
- participación sobre el cambio total.

El waterfall utiliza la contribución absoluta porque los porcentajes de períodos sucesivos son multiplicativos y no pueden sumarse directamente.


In [ ]:
def build_contribution_summary(
    fact: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    """Resume los cambios diarios para una autoridad y conserva el orden temporal."""
    rows = []

    for authority, group in fact.groupby(group_column, sort=False, dropna=False):
        group = group.sort_values("date")
        opening_rate = group["previous_exchange_rate_ars"].iloc[0]

        # Para la primera autoridad de la serie no existe una cotización anterior.
        if pd.isna(opening_rate):
            opening_rate = group["exchange_rate_ars"].iloc[0]

        variation_ars = group["daily_change_ars"].sum()
        closing_rate = group["exchange_rate_ars"].iloc[-1]

        rows.append(
            {
                group_column: authority,
                "sequence": len(rows) + 1,
                "first_quote_date": group["date"].min(),
                "last_quote_date": group["date"].max(),
                "quote_observations": len(group),
                "opening_reference_ars": float(opening_rate),
                "closing_rate_ars": float(closing_rate),
                "variation_ars": float(variation_ars),
                "variation_pct_from_opening": (
                    float(variation_ars / opening_rate * 100)
                    if opening_rate != 0
                    else pd.NA
                ),
            }
        )

    summary = pd.DataFrame(rows)
    total_change = fact["daily_change_ars"].sum()
    summary["share_of_total_change_pct"] = (
        summary["variation_ars"] / total_change * 100
        if total_change != 0
        else pd.NA
    )
    summary["product_refreshed_at_utc"] = pd.Timestamp.now(tz="UTC")
    return summary

In [ ]:
def validate_contribution_summary(
    fact: pd.DataFrame,
    summary: pd.DataFrame,
    label: str,
    tolerance: float = 1e-8,
) -> None:
    """Verifica que las contribuciones agregadas lleguen al nivel actual."""
    initial_rate = fact["exchange_rate_ars"].iloc[0]
    current_rate = fact["exchange_rate_ars"].iloc[-1]
    reconstructed_rate = initial_rate + summary["variation_ars"].sum()

    if abs(current_rate - reconstructed_rate) > tolerance:
        raise ValueError(
            f"El waterfall de {label} no reconcilia con la cotización actual."
        )

## 9. Persistencia de la capa Gold

Las tres salidas son snapshots derivados. `overwrite` mantiene una versión coherente entre el detalle y los agregados; `VACUUM` elimina archivos de datos obsoletos después del período de seguridad.


In [ ]:
def save_snapshot_as_delta(df: pd.DataFrame, data_path: str) -> None:
    """Guarda un DataFrame como snapshot Delta sin incluir el índice."""
    arrow_table = pa.Table.from_pandas(df, preserve_index=False)
    write_deltalake(
        data_path,
        arrow_table,
        mode="overwrite",
        storage_options=storage_options,
    )


def vacuum_delta_table(data_path: str) -> list[str]:
    """Elimina archivos obsoletos respetando la retención configurada."""
    delta_table = DeltaTable(data_path, storage_options=storage_options)
    deleted_files = delta_table.vacuum(
        retention_hours=VACUUM_RETENTION_HOURS,
        dry_run=False,
        enforce_retention_duration=True,
    )
    print(f"VACUUM {data_path}: {len(deleted_files)} archivos eliminados")
    return deleted_files

## 10. Orquestación del data product

La función siguiente concentra lectura, validación, transformación, reconciliación y carga. Devuelve los DataFrames para inspección y visualización sin volver a leerlos desde MinIO.


In [ ]:
def run_data_product() -> dict[str, Any]:
    """Construye, valida y publica el data product Gold."""
    run_started_at = pd.Timestamp.now(tz="UTC")

    usd_quotes, government_daily = extract_source_products()
    validate_source_products(usd_quotes, government_daily)

    daily = build_usd_government_daily(usd_quotes, government_daily)
    validate_daily_product(daily)

    by_president = build_contribution_summary(daily, "president")
    by_minister = build_contribution_summary(daily, "economy_minister")
    validate_contribution_summary(daily, by_president, "presidentes")
    validate_contribution_summary(daily, by_minister, "ministros")

    outputs = [
        (daily, USD_GOVERNMENT_DAILY_PATH),
        (by_president, USD_BY_PRESIDENT_PATH),
        (by_minister, USD_BY_MINISTER_PATH),
    ]

    for dataframe, data_path in outputs:
        save_snapshot_as_delta(dataframe, data_path)
        vacuum_delta_table(data_path)

    run_finished_at = pd.Timestamp.now(tz="UTC")
    metrics = {
        "run_started_at": run_started_at,
        "run_finished_at": run_finished_at,
        "duration_seconds": (run_finished_at - run_started_at).total_seconds(),
        "first_quote_date": daily["date"].min(),
        "last_quote_date": daily["date"].max(),
        "initial_exchange_rate_ars": daily["exchange_rate_ars"].iloc[0],
        "current_exchange_rate_ars": daily["exchange_rate_ars"].iloc[-1],
        "total_change_ars": daily["daily_change_ars"].sum(),
        "daily_rows": len(daily),
        "presidents": len(by_president),
        "economy_ministers": len(by_minister),
    }

    print("Data product Gold publicado correctamente.")
    return {
        "daily": daily,
        "by_president": by_president,
        "by_minister": by_minister,
        "metrics": metrics,
    }

In [ ]:
product = run_data_product()

usd_government_daily = product["daily"]
usd_by_president = product["by_president"]
usd_by_minister = product["by_minister"]
product_metrics = product["metrics"]

display(pd.Series(product_metrics, name="value").to_frame())

## 11. Inspección del producto

Antes de visualizar, se muestran las tablas que alimentan cada gráfico. Estas mismas tablas pueden conectarse desde Power BI, Tableau u otra herramienta compatible con el almacenamiento utilizado.


In [ ]:
display(
    usd_government_daily[
        [
            "date",
            "exchange_rate_ars",
            "daily_change_ars",
            "president",
            "economy_minister",
        ]
    ].tail(10)
)

display(usd_by_president)
display(usd_by_minister)

## 12. Waterfall charts

Los aumentos del tipo de cambio se muestran en rojo y las disminuciones en verde. Los extremos azules representan los niveles inicial y actual. Al pasar el cursor por una gestión se observan su contribución, sus fechas y la cantidad de cotizaciones incluidas.


In [ ]:
def create_waterfall_chart(
    summary: pd.DataFrame,
    authority_column: str,
    title: str,
    initial_rate: float,
    current_rate: float,
    width: int = 1200,
) -> go.Figure:
    """Construye un waterfall reconciliado desde el nivel inicial al actual."""
    labels = summary[authority_column].astype(str).tolist()
    contributions = summary["variation_ars"].astype(float).tolist()

    x_values = ["Nivel inicial"] + labels + ["Nivel actual"]
    y_values = [float(initial_rate)] + contributions + [0.0]
    measures = ["absolute"] + ["relative"] * len(contributions) + ["total"]

    custom_rows = [["", "", ""]]
    custom_rows.extend(
        [
            [
                row.first_quote_date.strftime("%Y-%m-%d"),
                row.last_quote_date.strftime("%Y-%m-%d"),
                int(row.quote_observations),
            ]
            for row in summary.itertuples(index=False)
        ]
    )
    custom_rows.append(["", "", ""])

    display_values = [float(initial_rate)] + contributions + [float(current_rate)]
    text_values = [f"$ {value:,.2f}" for value in display_values]

    figure = go.Figure(
        go.Waterfall(
            name="Variación USD",
            orientation="v",
            measure=measures,
            x=x_values,
            y=y_values,
            text=text_values,
            textposition="outside",
            customdata=custom_rows,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Valor/contribución: %{text}<br>"
                "Primera cotización: %{customdata[0]}<br>"
                "Última cotización: %{customdata[1]}<br>"
                "Observaciones: %{customdata[2]}"
                "<extra></extra>"
            ),
            connector={"line": {"color": "#94A3B8", "dash": "dot"}},
            increasing={"marker": {"color": "#DC2626"}},
            decreasing={"marker": {"color": "#16A34A"}},
            totals={"marker": {"color": "#2563EB"}},
        )
    )

    figure.update_layout(
        title={"text": title, "x": 0.02},
        xaxis_title="Gestión en orden cronológico",
        yaxis_title="Pesos argentinos por USD",
        template="plotly_white",
        showlegend=False,
        width=width,
        height=650,
        margin={"l": 70, "r": 40, "t": 90, "b": 180},
    )
    figure.update_xaxes(tickangle=-45, automargin=True)
    return figure

In [ ]:
initial_rate = usd_government_daily["exchange_rate_ars"].iloc[0]
current_rate = usd_government_daily["exchange_rate_ars"].iloc[-1]

president_waterfall = create_waterfall_chart(
    summary=usd_by_president,
    authority_column="president",
    title="Variación del dólar oficial por presidencia",
    initial_rate=initial_rate,
    current_rate=current_rate,
    width=1300,
)
president_waterfall.show()

In [ ]:
minister_waterfall = create_waterfall_chart(
    summary=usd_by_minister,
    authority_column="economy_minister",
    title="Variación del dólar oficial por gestión económica",
    initial_rate=initial_rate,
    current_rate=current_rate,
    width=1900,
)
minister_waterfall.show()

## 13. Consumo desde una herramienta de BI

Para visualización externa se recomienda consumir directamente las tablas agregadas:

- `usd_variation_by_president`
- `usd_variation_by_economy_minister`

Campos principales del waterfall:

| Campo | Uso visual |
|---|---|
| `sequence` | Orden cronológico |
| `president` / `economy_minister` | Categoría |
| `variation_ars` | Tamaño y dirección del bar |
| `opening_reference_ars` | Tooltip de apertura |
| `closing_rate_ars` | Tooltip de cierre |
| `first_quote_date` / `last_quote_date` | Contexto temporal |

En Power BI o Tableau deben agregarse dos extremos calculados: el nivel inicial como barra absoluta y el nivel actual como total. Las contribuciones intermedias utilizan `variation_ars` ordenada por `sequence`.


## 14. Conclusiones y aprendizajes

Esta tercera notebook completa un flujo de datos por capas:

- separa extracción, contexto y consumo analítico;
- reutiliza productos upstream en lugar de repetir llamadas a APIs;
- aplica contratos de datos y validaciones antes del join;
- documenta una regla de atribución reproducible;
- demuestra reconciliación financiera entre el nivel inicial, las variaciones y el nivel actual;
- publica datasets Gold diseñados para visualización;
- diferencia correlación temporal de causalidad.

Para un post de LinkedIn, el eje narrativo puede ser: **cómo convertir dos pipelines independientes en un data product gobernado, reconciliado y listo para decisiones**, mostrando la arquitectura Bronze–Silver–Gold y los dos waterfall charts como resultado final.
